In [ ]:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

fact_sales_fe=pd.read_csv(r"C:\Users\RohanS\Documents\retail project\data retail processed\fact_sales_fe.csv")
df = fact_sales_fe.copy()



df["low_review"] = (
    df["review_score"] <= 2
).astype(int)



features = [
    "price",
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

model_df = df[
    features + ["low_review"]
].copy()



model_df = model_df.dropna()



X = model_df[features]

y = model_df["low_review"]


print("Dataset shape:", model_df.shape)

print("\nTarget distribution:")
print(
    y.value_counts()
)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)




y_pred = model.predict(
    X_test
)




print("\nAccuracy:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


importance = pd.DataFrame({

    "feature": features,

    "importance": model.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)


print("\nFeature Importance:")
print(importance)



prediction_df = df[
    features
].copy()

prediction_df = prediction_df.fillna(
    prediction_df.median(numeric_only=True)
)


df["low_review_probability"] = (
    model.predict_proba(
        prediction_df
    )[:, 1]
)


df["review_risk"] = pd.cut(

    df["low_review_probability"],

    bins=[
        -np.inf,
        0.30,
        0.60,
        np.inf
    ],

    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)



review_risk = df[
    [
        "order_id",
        "product_id",
        "seller_id",
        "low_review_probability",
        "review_risk"
    ]
].copy()


review_risk["low_review_probability"] = (
    review_risk["low_review_probability"]
    .round(3)
)


print("\nReview risk:")
print(
    review_risk.head(20)
)

print("\nRisk distribution:")
print(
    review_risk["review_risk"]
    .value_counts()
)



review_risk.to_csv(
    "C:/Users/RohanS/Desktop/customer_review_predictions_8_3.csv",
    index=False
)

importance.to_csv(
    "C:/Users/RohanS/Desktop/review_feature_importance_8_3.csv",
    index=False
)

